In [ ]:
import sys; sys.path.append('..'); sys.path.append('../..'); sys.path.append('../../gmsh/')

In [ ]:
from periodic_simulation_setup import *

In [ ]:
import json

In [ ]:
h = 5
w = 5
avg_len = 0.1

In [ ]:
import importlib
importlib.reload(periodic_unit_helper)

In [ ]:
from periodic_unit_helper import *

In [ ]:
import pattern_generator_using_gmsh

In [ ]:
sys.path.append('../../gmsh/')

In [ ]:
import mesher_helper

In [ ]:
importlib.reload(pattern_generator_using_gmsh)

In [ ]:
import ipywidgets
import ipywidgets as widgets


In [ ]:
from ipywidgets import interact

In [ ]:
angle = 10
def plot(angle):
    r = 2.5 / np.sqrt(2) * 0.9
    r = 1.5

    print("angle", angle, "radius: ", r)
    dash_point = np.array([np.cos(angle / 180 * np.pi), np.sin(angle / 180 * np.pi)]) * r + np.array([0, 0])

    w = 5
    h = 5

    boundary_vxs = np.array([[-w/2., -h/2.], [w/2., -h/2.], [w/2., h/2.], [-w/2., h/2.]])
    boundary_lines = np.array([[0, 1], [1, 2], [2, 3], [3, 0]]) 


    # ipu, m, fusing_data = pattern_generator_using_gmsh.get_cosine_dash(h, 0.1, 0.1, amplitude=-0.5, dash_point = dash_point)

    ipu, m, marker = pattern_generator_using_gmsh.get_cosine_dash(h, 0.2, 0.2, amplitude=0.3, dash_point = dash_point)

    # visualization.plot_2d_mesh(m, pointList=marker, width=5, height=5)



    finalMarkers = np.where(np.array(marker) == 1)[0]
    m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, flip_orientation= 0)
    m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, axis = 1, flip_orientation= 1)

    fusedVtx = get_fusedVtx_using_markers(len(m.vertices()), finalMarkers)

    visualization.plot_2d_mesh(m, pointList=fusedVtx, width=5, height=5)



In [ ]:
len(m.vertices())

In [ ]:
interact(plot, angle=widgets.IntSlider(min=0, max=180, step=1, value=10));

In [ ]:
np.linspace(0, 3, 61)

In [ ]:
np.linspace(1, 2.4, 8)

In [ ]:
r = 2.5 / np.sqrt(2) * 0.9
r = 1

print("angle", angle, "radius: ", r)
dash_point = np.array([np.cos(angle / 180 * np.pi), np.sin(angle / 180 * np.pi)]) * r + np.array([0, 0])

w = 5
h = 5

boundary_vxs = np.array([[-w/2., -h/2.], [w/2., -h/2.], [w/2., h/2.], [-w/2., h/2.]])
boundary_lines = np.array([[0, 1], [1, 2], [2, 3], [3, 0]]) 


# ipu, m, fusing_data = pattern_generator_using_gmsh.get_cosine_dash(h, 0.1, 0.1, amplitude=-0.5, dash_point = dash_point)

ipu, m, marker = pattern_generator_using_gmsh.get_cosine_dash(h, 0.2, 0.2, amplitude=0.3, dash_point = dash_point)

# visualization.plot_2d_mesh(m, pointList=marker, width=5, height=5)



finalMarkers = np.where(np.array(marker) == 1)[0]
m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, flip_orientation= 0)
m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, axis = 1, flip_orientation= 1)

fusedVtx = get_fusedVtx_using_markers(len(m.vertices()), finalMarkers)

visualization.plot_2d_mesh(m, pointList=fusedVtx, width=5, height=5)



In [ ]:
len(m.vertices())

In [ ]:
ipu = inflation.InflatablePeriodicUnit(m, fusedVtx = fusedVtx, epsilon = 1e-9)

In [ ]:

from tri_mesh_viewer import TriMeshViewer
viewer = TriMeshViewer(ipu, width=768, height=640)
viewer.showWireframe(True)

In [ ]:
viewer.show()

In [ ]:
# Choose strategy for constraining rigid motion
fixedVars, hessianShift = list(periodic_unit_helper.get_center_fixedVars(ipu)) + [ipu.numVars() - 2], 0
fixedVars, hessianShift = [ipu.numVars() - 2, ipu.numVars() - 1], 1e-6
# fixedVars, hessianShift = [], 1e-6

ipu.sheet.setUseTensionFieldEnergy(True)
ipu.sheet.setUseHessianProjectedEnergy(False)

ipu.sheet.pressure = 0.4

In [ ]:
opts.niter = 2000
framerate = 1 # Update every 5 iterations
def cb(it):
    if it % framerate == 0:
        viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
cr = inflation.inflation_newton(ipu, fixedVars, opts, callback=cb, hessianShift = hessianShift)
cr.success

In [ ]:
az_ipu = get_az_ipu_from_ipu(ipu, m, fusedVtx, True)

In [ ]:
az_ipu = inflation.InflatableMidSurfacePeriodicUnit(m, fusedVtx, epsilon = 1e-5)
az_ipu.ipu.setVars(ipu.getVars())
az_ipu.ipu.sheet.setUseTensionFieldEnergy(True)
az_ipu.ipu.sheet.setUseHessianProjectedEnergy(False)
az_ipu.ipu.sheet.pressure = ipu.sheet.pressure

In [ ]:

from tri_mesh_viewer import TriMeshViewer
az_viewer = TriMeshViewer(az_ipu, width=768, height=640)
az_viewer.showWireframe(True)

In [ ]:
az_viewer.show()

In [ ]:
def az_cb(it):
    if it % framerate == 0:
        az_viewer.update(scalarField=utils.getStrains(az_ipu.ipu.sheet)[:, 0])

In [ ]:
fixedVars, hessianShift = [az_ipu.numVars() - 2, az_ipu.numVars() - 1], 1e-6
opts.niter = 400
az_optimizer = inflation.get_inflation_optimizer(az_ipu, fixedVars, opts, callback=az_cb, hessianShift = hessianShift)
cr = az_optimizer.optimize()

In [ ]:
benchmark.reset()
stiffness_values, sampled_alphas = visualize_sampled_bending_stiffness(az_ipu, 1000, az_optimizer, hessianShift = 1e-10, fixedVars = [])
benchmark.report()
min(stiffness_values), max(stiffness_values)

In [ ]:
import experiment_helper
import igl
from periodic_simulation_setup import *
import json

allowBending = False
useTFT = True
useMirror = False

name = 'cosine_curve_{}'.format("mirror" if useMirror else "no_mirror")
time_stamp = time.strftime("%Y_%m_%d_%H_%M")
result_folder = 'output/{}/{}'.format(name, time_stamp)
if not os.path.exists(result_folder):
    os.makedirs(result_folder)  

# pressure = 0.8
stiffness_pressure = 0.4
scale_factor_pressure = 0.01

amplitudes = np.linspace(0, 3, 61)

amp = amplitudes[14]
h = 5
avg_len = 0.1
ipu, new_pts, new_edges, m, marker = periodic_unit_helper.get_cosine_curve(h, avg_len, amplitude=amp)
finalMarkers = np.where(np.array(marker) == 1)[0]
m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers)
m, finalMarkers = periodic_unit_helper.shift_and_merge_2D_periodic_mesh(m, finalMarkers, axis = 1, flip_orientation= 1 if useMirror else -1)

fusedVtx = get_fusedVtx_using_markers(len(m.vertices()), finalMarkers)
ipu = inflation.InflatablePeriodicUnit(m, fusedVtx = fusedVtx, epsilon = 1e-9)

In [ ]:
# experiment_helper.run_experiment(ipu, m, fusedVtx, stiffness_pressure, scale_factor_pressure, name, '%.2f'%amp, allowBending, result_folder, useTFT = useTFT)


In [ ]:
low_pressure_tag = "low_pressure"
high_pressure_tag = "high_pressure"


In [ ]:
disableFusedRegionTFT = False
useTFT = True

In [ ]:
variable = '%.2f'%amp

In [ ]:
viewer = TriMeshViewer(ipu, width=768, height=640)
viewer.showWireframe(True)

# Choose strategy for constraining rigid motion
fixedVars, hessianShift = periodic_unit_helper.get_center_fixedVars(ipu), 0
if not allowBending:
    fixedVars, hessianShift = [ipu.numVars() - 2, ipu.numVars() - 1], 1e-6
else:
    fixedVars, hessianShift = [], 1e-6

ipu.sheet.setUseTensionFieldEnergy(useTFT)
ipu.sheet.setUseHessianProjectedEnergy(False)
if (disableFusedRegionTFT):
    ipu.sheet.disableFusedRegionTensionFieldTheory(False)
ipu.sheet.pressure = stiffness_pressure

benchmark.reset()
print(allowBending, stiffness_pressure, hessianShift, fixedVars)

opts.niter = 500
cr = inflation.inflation_newton(ipu, fixedVars, opts, callback=None, hessianShift = hessianShift)
viewer.update(scalarField=utils.getStrains(ipu.sheet)[:, 0])
benchmark.report()

In [ ]:
az_ipu = get_az_ipu_from_ipu(ipu, m, fusedVtx, useTFT, disableFusedRegionTFT)
if not allowBending:
    fixedVars, hessianShift = [az_ipu.numVars() - 2, az_ipu.numVars() - 1], 1e-6
else:
    fixedVars, hessianShift = [], 1e-6
opts.niter = 1000
az_optimizer = inflation.get_inflation_optimizer(az_ipu, fixedVars, opts, callback=None, hessianShift = hessianShift)
cr = az_optimizer.optimize()

In [ ]:
# stiffness_values

In [ ]:
if not allowBending:
    stiffness_shift = 1e-15
    for i in range(15):
        try:
            stiffness_values, sampled_alphas = visualize_sampled_bending_stiffness(az_ipu, 1000, az_optimizer, hessianShift = stiffness_shift, fixedVars = [], filename = "{}/stiffness_{}_{}.png".format(result_folder, name, variable))
            np.save("{}/stiffness_values_{}_{}.npy".format(result_folder, name, variable), stiffness_values)
            np.save("{}/sampled_alphas_{}_{}.npy".format(result_folder, name, variable), sampled_alphas)
            break
        except:
            print("failed to compute stiffness with shift ", stiffness_shift)
            stiffness_shift *= 10
    print("Solved using stiffness shift: ", stiffness_shift)

In [ ]:
plt.hist(utils.getStrains(ipu.sheet)[:, 0])

In [ ]:
viewer.show()

In [ ]:
stiffness_values = None
benchmark.reset()
stiffness_shift = 1e-15
for i in range(15):
    try:
        stiffness_values = inflation.get_equilibrium_sensitivity(az_ipu, 0, az_optimizer, hessianShift = stiffness_shift, fixedVars = [])
        break
    except:
        print("failed to compute stiffness with shift ", stiffness_shift)
    stiffness_shift *= 10
benchmark.report()


In [ ]:
modes = np.array([stiffness_values]).reshape(len(stiffness_values), 1)
lambdas = [1]

In [ ]:
dE_dkappa_dkappa = modes[-2][0]

In [ ]:
dE_dkappa_dkappa

In [ ]:
modes[-2][0] = 1
modes[-1][0] = 0

In [ ]:
az_ipu.numVars()

In [ ]:
stiffness_values.shape

In [ ]:
import mode_viewer, importlib
mview = mode_viewer.ModeViewer(az_ipu, modes, lambdas, amplitude=0.01)
mview.show()